# Hardware test — instance scoping (`feat/instance-scoped-api`)

Exercises the thread-safe instance scoping features on real IviumSoft instances:

1. Selection tracking (`Core.get_selected_instance()`)
2. `Pyvium.on_instance(n)` — atomic select / run / restore (incl. nesting and exceptions)
3. `Pyvium.device(n)` — instance-bound handles
4. `get_active_iviumsoft_instances()` — no longer hijacks the selection
5. Two-thread stress test — commands never land on the wrong instance

**Prerequisites**
- Windows machine with IviumSoft installed (x64 Python — the bundled driver DLLs are x86/x64 only)
- Run from the repo on branch `feat/instance-scoped-api`
- Ideally **2+ IviumSoft instances**; cell 3 can launch them for you
- For full coverage (serial-number checks, stress test): **2+ devices connected**, one per instance.
  Without devices the notebook still verifies tracking, scoping and restore via the status calls.

In [ ]:
import sys
from pathlib import Path

# Import the local dev copy (this branch), not a pip-installed pyvium.
# Walk up until the repo root (the directory holding pyproject.toml) so the
# notebook works regardless of how deep it sits inside notebooks/.
repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists():
    if repo_root.parent == repo_root:
        raise RuntimeError("repo root not found — run this notebook from inside the pyvium repo")
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pyvium
from pyvium import Core, Pyvium

print("pyvium imported from:", pyvium.__file__)
assert hasattr(Pyvium, "on_instance"), "on_instance missing — wrong branch or wrong pyvium import?"
assert hasattr(Pyvium, "device"), "device() missing — wrong branch or wrong pyvium import?"

### (Optional) Launch two IviumSoft instances
Skip this cell if the instances are already running. Adjust `IVIUMSOFT_EXE` if needed.

In [ ]:
import subprocess
import time

IVIUMSOFT_EXE = r"C:\IviumStat\IviumSoft.exe"

launched = []
for _ in range(2):
    launched.append(subprocess.Popen(
        [IVIUMSOFT_EXE],
        stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        close_fds=True,
    ))
    time.sleep(2)

print("Launched PIDs:", [p.pid for p in launched])

### Open the driver and discover instances

In [ ]:
Pyvium.open_driver()
active = Pyvium.get_active_iviumsoft_instances()
print("Active IviumSoft instances:", active)

assert len(active) >= 1, "No IviumSoft instance running — launch at least one and re-run"
if len(active) < 2:
    print("WARNING: only one instance — scoping tests run, but cross-instance checks are weak")

INSTANCE_A = active[0]
INSTANCE_B = active[1] if len(active) > 1 else active[0]
print(f"Using INSTANCE_A={INSTANCE_A}, INSTANCE_B={INSTANCE_B}")

## 1. Selection tracking
The DLL has no getter for the selected instance; the branch tracks every `IV_selectdevice`
in `CoreBase`. Verify the tracker follows explicit selections.

In [ ]:
Pyvium.select_iviumsoft_instance(INSTANCE_A)
assert Core.get_selected_instance() == INSTANCE_A

Pyvium.select_iviumsoft_instance(INSTANCE_B)
assert Core.get_selected_instance() == INSTANCE_B

Pyvium.select_iviumsoft_instance(INSTANCE_A)
assert Core.get_selected_instance() == INSTANCE_A
print("PASS — tracker follows explicit selection")

## 2. `on_instance` — select, run, restore
Inside the block the target instance is selected; afterwards the previous selection is
restored — including when the block raises, and when blocks are nested.

In [ ]:
# Basic: select + restore
with Pyvium.on_instance(INSTANCE_B):
    assert Core.get_selected_instance() == INSTANCE_B
    print(f"  inside block: instance {INSTANCE_B}, status = {Pyvium.get_device_status()}")
assert Core.get_selected_instance() == INSTANCE_A

# Nesting
with Pyvium.on_instance(INSTANCE_B):
    with Pyvium.on_instance(INSTANCE_A):
        assert Core.get_selected_instance() == INSTANCE_A
    assert Core.get_selected_instance() == INSTANCE_B
assert Core.get_selected_instance() == INSTANCE_A

# Restore on exception
try:
    with Pyvium.on_instance(INSTANCE_B):
        raise RuntimeError("deliberate")
except RuntimeError:
    pass
assert Core.get_selected_instance() == INSTANCE_A

print("PASS — on_instance selects, nests, and restores (also on exception)")

## 3. `Pyvium.device(n)` handles
Interleave calls on two handles without ever selecting manually. Every call scopes itself;
the global selection is untouched afterwards. With devices connected, the serial numbers
prove each call hit the right instance.

In [ ]:
device_a = Pyvium.device(INSTANCE_A)
device_b = Pyvium.device(INSTANCE_B)

before = Core.get_selected_instance()

# Interleaved status calls — no manual selection anywhere
print(f"instance {INSTANCE_A} status:", device_a.get_device_status())
print(f"instance {INSTANCE_B} status:", device_b.get_device_status())
print(f"instance {INSTANCE_A} status:", device_a.get_device_status())

assert Core.get_selected_instance() == before, "handles must not leak selection changes"

# Serial numbers (needs connected devices)
serials = {}
for number, handle in [(INSTANCE_A, device_a), (INSTANCE_B, device_b)]:
    try:
        serials[number] = handle.get_device_serial_number()
        print(f"instance {number} serial: {serials[number]}")
    except Exception as error:  # no device connected on this instance
        print(f"instance {number}: serial unavailable ({type(error).__name__})")

if len(serials) == 2 and INSTANCE_A != INSTANCE_B:
    assert serials[INSTANCE_A] != serials[INSTANCE_B], (
        "same serial from both handles — scoping failed or both instances share a device")
    print("PASS — each handle reads its own device's serial")
else:
    print("PASS (partial) — handles scope correctly; connect 2 devices for the serial cross-check")

## 4. Scanner no longer hijacks the selection
Before this branch, `get_active_iviumsoft_instances()` left the *first active* instance
selected. Now it restores whatever was selected before the scan.

In [ ]:
Pyvium.select_iviumsoft_instance(INSTANCE_B)

rescanned = Pyvium.get_active_iviumsoft_instances()

assert Core.get_selected_instance() == INSTANCE_B, "scan must restore the previous selection"
print("active instances:", rescanned)
print(f"PASS — selection still on instance {INSTANCE_B} after the scan")

Pyvium.select_iviumsoft_instance(INSTANCE_A)  # leave a known state

## 5. Two-thread stress test
The race this branch eliminates: thread A selects its instance, thread B switches the
selection before A's command fires. Two threads hammer two handles; with devices
connected, every serial read must match the instance the handle is bound to.
Requires `INSTANCE_A != INSTANCE_B` and a device on each — skips otherwise.

In [ ]:
import threading

ROUNDS = 50

if INSTANCE_A == INSTANCE_B or len(serials) < 2:
    print("SKIPPED — needs two instances, each with a connected device (run cell 3 checks first)")
else:
    mismatches = []
    failures = []

    def worker(instance_number, expected_serial):
        handle = Pyvium.device(instance_number)
        for round_number in range(ROUNDS):
            try:
                handle.get_device_status()
                observed = handle.get_device_serial_number()
                if observed != expected_serial:
                    mismatches.append((instance_number, round_number, observed))
            except Exception as error:
                failures.append((instance_number, round_number, repr(error)))

    threads = [
        threading.Thread(target=worker, args=(INSTANCE_A, serials[INSTANCE_A])),
        threading.Thread(target=worker, args=(INSTANCE_B, serials[INSTANCE_B])),
    ]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()

    print(f"{2 * ROUNDS} scoped call pairs, {len(mismatches)} wrong-instance reads, {len(failures)} errors")
    if mismatches:
        print("MISMATCHES (instance, round, observed serial):", mismatches[:10])
    if failures:
        print("ERRORS:", failures[:10])
    assert not mismatches, "FAIL — a command landed on the wrong instance"
    print("PASS — no cross-instance leakage under concurrency")

## Cleanup
Closes the driver. The second cell also gracefully closes the instances launched by this
notebook (window-close message, same as the archive notebook — refuses nothing, so make
sure no measurement is running).

In [ ]:
Pyvium.close_driver()
print("driver closed")

In [ ]:
# Optional: close the IviumSoft instances launched earlier in this notebook
for process in launched:
    subprocess.run(
        ["powershell", "-Command", f"(Get-Process -Id {process.pid}).CloseMainWindow()"],
        capture_output=True, check=False,
    )
    print(f"sent close to PID {process.pid}")